# Modelos supervisados
Regresión para estimar Y1 y Y2 y clasificación de calefacción. Ejecuta primero la preparación.

Se agrupa por las 12 geometrías (X1-X5) para mantener juntas sus variantes. Se reservan 3 geometrías para prueba y la elección de algoritmos usa solamente el entrenamiento.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Carpeta del proyecto:", ROOT.name)


Carpeta del proyecto: REC_ECBD_2026_Gonzalez_Ulises


## 1. Entrenar, comparar y ajustar
El código en src/modelos_supervisados.py utiliza Pipeline para ajustar el escalado y OneHotEncoder dentro de cada partición. GroupKFold con tres particiones compara algoritmos y parámetros. La clasificación usa un umbral fijo de 25 para Y1; es una regla didáctica que requiere revisión con el docente.

In [2]:
from src.modelos_supervisados import ajustar_modelos
resultados = ajustar_modelos()

Separación temporal 70/30: 537 entrenamiento / 231 prueba.
Regresión seleccionada mediante validación: Ridge {'modelo__alpha': 10.0}
          modelo objetivo    MAE    RMSE      R2
           Ridge       Y1 2.2025  2.9852  0.9186
           Ridge       Y2 2.3316  3.2475  0.8905
Referencia_media       Y1 9.7010 11.1841 -0.1426
Referencia_media       Y2 9.0216 10.2140 -0.0834
Clasificación seleccionada mediante validación: Logistica {'modelo__C': 0.1}
            modelo  accuracy  precision_macro  recall_macro  F1_macro
         Logistica    1.0000           1.0000           1.0    1.0000
Referencia_mayoria    0.5195           0.2597           0.5    0.3419
Matriz de confusión [baja, alta]: [[120, 0], [0, 111]]


## 2. Revisar la separación
Las geometrías de prueba no aparecen en el entrenamiento. Los identificadores, grupos y cargas objetivo no se usan como entradas.

In [3]:
print("Geometrías de entrenamiento:", resultados["geometrias_entrenamiento"])
print("Geometrías de prueba:", resultados["geometrias_prueba"])
print("Geometrías compartidas:", resultados["geometrias_compartidas"])
print("Filas de entrenamiento:", resultados["entrenamiento_filas"])
print("Filas de prueba:", resultados["prueba_filas"])

Geometrías de entrenamiento: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Geometrías de prueba: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Geometrías compartidas: 12
Filas de entrenamiento: 537
Filas de prueba: 231


## 3. Comparar los errores con una referencia simple
MAE y RMSE se interpretan en la escala de la carga; no son porcentajes. R2 puede ser negativo. F1 macro da el mismo peso a ambas clases.

In [4]:
print(pd.read_csv(ROOT / "docs/metricas_regresion.csv").round(4).to_string(index=False))
print(pd.read_csv(ROOT / "docs/metricas_clasificacion.csv").round(4).to_string(index=False))
print("Matriz [baja, alta]:", resultados["clasificacion"]["matriz_confusion_baja_alta"])

          modelo objetivo    MAE    RMSE      R2
           Ridge       Y1 2.2025  2.9852  0.9186
           Ridge       Y2 2.3316  3.2475  0.8905
Referencia_media       Y1 9.7010 11.1841 -0.1426
Referencia_media       Y2 9.0216 10.2140 -0.0834
            modelo  accuracy  precision_macro  recall_macro  F1_macro
         Logistica    1.0000           1.0000           1.0    1.0000
Referencia_mayoria    0.5195           0.2597           0.5    0.3419
Matriz [baja, alta]: [[120, 0], [0, 111]]


![Regresión en prueba](../docs/figuras/04_regresion_prueba.png)

![Matriz de confusión](../docs/figuras/03_matriz_confusion.png)

## 4. Cargar un modelo y predecir
Este ejemplo utiliza un caso reservado para prueba. El valor real se muestra para compararlo con la estimación.

In [5]:
import joblib
from src.preparacion_datos import ENTRADAS
caso = pd.read_csv(ROOT / "data/processed/predicciones_prueba.csv").iloc[[0]]
modelo = joblib.load(ROOT / "modelos/modelo_regresion_cargas.joblib")
print("Entradas:")
print(caso[ENTRADAS].to_string(index=False))
print("Cargas reales:", caso[["Y1", "Y2"]].to_numpy()[0])
print("Cargas predichas:", modelo.predict(caso[ENTRADAS])[0].round(2))

Entradas:
  X1    X2    X3    X4  X5  X6  X7  X8
0.86 588.0 294.0 147.0 7.0   3 0.4   1
Cargas reales: [32.39 30.66]
Cargas predichas: [32.4  33.84]


## Interpretación pendiente
Esta es una evaluación inicial con una sola separación y datos simulados. Hay que revisar estabilidad en otras separaciones por geometría y los errores de clasificación. Los resultados no prueban ahorros en edificios reales.

Referencias: https://doi.org/10.24432/C51307 y https://scikit-learn.org/stable/modules/cross_validation.html.